# Minimum QLoRA fine-tune (Colab / ~T4 16GB)

Trains a tiny companion-style chat adapter on **Qwen2.5-1.5B-Instruct** (4-bit + LoRA).

## Setup
1. Open this file in [Google Colab](https://colab.research.google.com/)
2. Runtime → Change runtime type → **GPU** (T4 is fine)
3. Run all cells
4. Download `min-qlora.zip` at the end

This is the cloud version of `finetune/train_min.py` (no local NVIDIA GPU needed).

## 1) Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime type → GPU"
props = torch.cuda.get_device_properties(0)
print(f"PyTorch {torch.__version__}")
print(f"GPU: {props.name} ({props.total_memory/1024**3:.1f} GB)")

## 2) Install packages

In [ ]:
# Pin protobuf for Colab preinstalled Google libs (need protobuf < 6)
%pip -q install -U "transformers" "datasets" "accelerate" "peft" "trl" "bitsandbytes" "sentencepiece" "protobuf>=5.26.1,<6"
import google.protobuf
print("protobuf", google.protobuf.__version__)


## 3) Create tiny training data

In [ ]:
from pathlib import Path

DATA = Path("/content/min_chat.jsonl")
OUT = Path("/content/min-qlora")
OUT.mkdir(parents=True, exist_ok=True)

DATA.write_text("""\
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"Hi"},{"role":"assistant","content":"Hi! I'm here. What would you like to talk about?"}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"How are you?"},{"role":"assistant","content":"I'm doing well, thanks for asking! How about you?"}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"What can you do?"},{"role":"assistant","content":"I can chat with you, answer simple questions, and keep you company on your desktop."}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"Tell me a short joke"},{"role":"assistant","content":"Why do programmers prefer dark mode? Because light attracts bugs!"}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"Good morning"},{"role":"assistant","content":"Good morning! Hope you have a smooth day. Need anything?"}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"I'm tired"},{"role":"assistant","content":"Rest if you can. A short break helps more than pushing through. I'm here when you're ready."}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"Thanks"},{"role":"assistant","content":"Anytime! Happy to help."}]}
{"messages":[{"role":"system","content":"You are a friendly desktop companion. Keep replies short and warm."},{"role":"user","content":"Bye"},{"role":"assistant","content":"Bye! Come chat again whenever you like."}]}
""".strip()
+ "\n",
encoding="utf-8",
)
print("Wrote", DATA, "lines:", sum(1 for _ in DATA.open()))

## 4) Train (1.5B QLoRA)

In [ ]:
import inspect
import json
import torch
from datasets import Dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

rows = [json.loads(line) for line in DATA.read_text(encoding="utf-8").splitlines() if line.strip()]
print("Examples:", len(rows))

# T4 (Colab default) does not support BF16 GradScaler — keep compute in FP16
# and disable AMP to avoid: NotImplementedError ... BFloat16
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print("GPU capability:", torch.cuda.get_device_capability(0), "use_bf16=", use_bf16)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)
model.config.use_cache = False

def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(rows).map(to_text)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# On T4: turn AMP off (fp16 GradScaler + bf16 grads crashes).
# On Ampere+: use bf16 without GradScaler.
sft_kwargs = dict(
    output_dir=str(OUT),
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="epoch",
    fp16=False,
    bf16=use_bf16,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    report_to=[],
    dataset_text_field="text",
    packing=False,
)
sig = inspect.signature(SFTConfig.__init__).parameters
if "max_length" in sig:
    sft_kwargs["max_length"] = 512
elif "max_seq_length" in sig:
    sft_kwargs["max_seq_length"] = 512

sft_args = SFTConfig(**sft_kwargs)

trainer_kwargs = dict(
    model=model,
    args=sft_args,
    train_dataset=dataset,
    peft_config=peft_config,
)
trainer_sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in trainer_sig:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

# Ensure trainable LoRA weights are not bf16 on T4
if not use_bf16:
    for _, p in trainer.model.named_parameters():
        if p.requires_grad and p.dtype == torch.bfloat16:
            p.data = p.data.to(torch.float32)

trainer.train()
trainer.save_model(str(OUT))
tokenizer.save_pretrained(str(OUT))
print("Saved adapter to", OUT)

## 5) Quick smoke test

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
ft = PeftModel.from_pretrained(base, str(OUT))
ft.eval()

messages = [
    {"role": "system", "content": "You are a friendly desktop companion. Keep replies short and warm."},
    {"role": "user", "content": "Hi"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(ft.device)
with torch.no_grad():
    out = ft.generate(**inputs, max_new_tokens=64, do_sample=True, temperature=0.7)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## 6) Download adapter zip

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/min-qlora"
shutil.make_archive(zip_path, "zip", str(OUT))
files.download("/content/min-qlora.zip")
print("Downloaded min-qlora.zip — keep this for later Ollama / local merge steps.")